In [1]:
import sys
sys.path.append("../")

import pandas as pd

from src.models import load_model, log_experiment, test_prediction

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [6]:
df = pd.read_csv("../data/processed/04_default_credit_features.csv")

df = df.drop(columns=["ID"], errors="ignore")

TARGET = "default payment next month"

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [8]:
model_paths = {
    "LogisticRegression": "../models/LogisticRegression.joblib",
    "DecisionTree": "../models/DecisionTree.joblib",
    "RandomForest": "../models/RandomForest.joblib",
    "SVM": "../models/SVM.joblib",
    "KNN": "../models/KNN.joblib"
}

for model_name, model_path in model_paths.items():
    print(f"Registrando {model_name}...")

    modelo = load_model(model_path)

    y_pred = modelo.predict(X_test)

    if hasattr(modelo, "predict_proba"):
        y_score = modelo.predict_proba(X_test)[:, 1]
    else:
        y_score = y_pred

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_score)
    }

    log_experiment(
        model_name=model_name,
        model=modelo.named_steps["clf"],
        metrics=metrics
    )

print("✅ CSV generado")

Registrando LogisticRegression...
Experimento registrado en ../models/experiments_log.csv
Registrando DecisionTree...
Experimento registrado en ../models/experiments_log.csv
Registrando RandomForest...
Experimento registrado en ../models/experiments_log.csv
Registrando SVM...
Experimento registrado en ../models/experiments_log.csv
Registrando KNN...


c:\Users\DIANA\miniforge3\envs\proyecto1\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


Experimento registrado en ../models/experiments_log.csv
✅ CSV generado


In [9]:
X_sample = X_test.head(5)

for model_name, model_path in model_paths.items():
    pred = test_prediction(model_path, X_sample)
    print(model_name, pred)

LogisticRegression [0 0 0 0 0]
DecisionTree [0 0 0 0 0]
RandomForest [0 0 0 0 0]
SVM [0 0 0 0 0]
KNN [0 0 0 0 0]


In [10]:
y_test.head(5)

6907     0
24575    0
26766    0
2156     1
3179     0
Name: default payment next month, dtype: int64

In [11]:
X_sample = X_test.sample(20, random_state=42)

for model_name, model_path in model_paths.items():
    pred = test_prediction(model_path, X_sample)
    print(model_name, pred)

LogisticRegression [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
DecisionTree [1 1 0 0 0 1 0 0 0 0 0 1 1 0 1 0 0 1 0 0]
RandomForest [1 1 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0]
SVM [0 1 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0]
KNN [0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0]


In [12]:
for model_name, model_path in model_paths.items():
    modelo = load_model(model_path)
    pred = modelo.predict(X_test)
    
    print(model_name)
    print(pd.Series(pred).value_counts())
    print()

LogisticRegression
0    5435
1     565
Name: count, dtype: int64

DecisionTree
0    4602
1    1398
Name: count, dtype: int64

RandomForest
0    5234
1     766
Name: count, dtype: int64

SVM
0    5254
1     746
Name: count, dtype: int64

KNN
0    5186
1     814
Name: count, dtype: int64

